## Minimizing Portfolio Risk (Multiple Assets)

In [5]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

In [6]:
df = pd.read_csv('../data/15stocks_price.csv')
df['date_gsheets'] = pd.to_datetime(df['date_gsheets'],
                                   format='mixed')
df.set_index('date_gsheets', inplace=True)

minimize(fun, x0, bounds=None, constraints=())

fun: the objective function to be minimized

x0: initial guess (weights of the initial securities)

bounds: weights of individual securities must be between 0 and 1

constraints: sum of individual weights is always equal to 1, we also specify the difference between the expected return of the portfolio and the target return

In [7]:
def getPortRisk(weights):
    returns = df.pct_change(1).dropna()
    num_stocks = len(returns.columns)
    vcv = returns.cov(ddof=1)

    var_p = np.dot(np.transpose(weights),
               np.dot(vcv, weights))
    sd_p = np.sqrt(var_p)
    sd_p_annual = sd_p * np.sqrt(250)

    return sd_p_annual

In [8]:
num_stocks = len(df.columns) #we have to set the date as index for this
init_weights = [1 / num_stocks] * num_stocks

In [9]:
bounds = tuple((0,1) for i in range(num_stocks))

In [11]:
constraints = ({'type' : 'eq',
        'fun' : lambda w : np.sum(w) - 1})

In [12]:
results = minimize(fun=getPortRisk,
                  x0=init_weights,
                  bounds=bounds,
                  constraints=constraints)

In [13]:
results

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 0.10593479369413471
           x: [ 4.798e-02  2.807e-01 ...  4.316e-02  1.247e-02]
         nit: 11
         jac: [ 1.060e-01  1.059e-01 ...  1.059e-01  1.063e-01]
        nfev: 176
        njev: 11
 multipliers: [ 1.059e-01]

In [24]:
print("Initial risk:", getPortRisk(init_weights))
print("Minimum risk:", results['fun'])

Initial risk: 0.13290599867193698
Minimum risk: 0.10593479369413471


In [28]:
optimized_weights = pd.DataFrame(results['x'])
optimized_weights.index = df.columns
optimized_weights.rename(
    columns={optimized_weights.columns[0]: 'weights'},
    inplace = True
)

In [29]:
optimized_weights

,weights
AAPL,4.797982e-02
KO,2.807425e-01
NFLX,7.531062e-03
BRK.B,1.292948e-01
DIS,3.542576e-02
IBM,8.404580e-02
VZ,1.628310e-01
WMT,1.728683e-01
GE,2.008729e-02
TSLA,1.897354e-18


In [30]:
optimized_weights['weights_rounded'] = \
    optimized_weights['weights'].apply(lambda x : round(x, 3))

In [31]:
optimized_weights

,weights,weights_rounded
AAPL,4.797982e-02,0.048
KO,2.807425e-01,0.281
NFLX,7.531062e-03,0.008
BRK.B,1.292948e-01,0.129
DIS,3.542576e-02,0.035
IBM,8.404580e-02,0.084
VZ,1.628310e-01,0.163
WMT,1.728683e-01,0.173
GE,2.008729e-02,0.020
TSLA,1.897354e-18,0.000


In [33]:
optimized_weights['weights_rounded'].sort_values(ascending=False).cumsum()

KO       0.281
WMT      0.454
VZ       0.617
BRK.B    0.746
IBM      0.830
AAPL     0.878
UN       0.921
DIS      0.956
GE       0.976
V        0.988
NFLX     0.996
AMZN     1.000
TSLA     1.000
MA       1.000
MSFT     1.000
Name: weights_rounded, dtype: float64

When we minimize the risk of a portfolio, we will be investing in securities that have relatively lower risk.